## Self notes — running log

Short observations only. One markdown cell per note. Add a code cell underneath if a demo is genuinely useful — most of the time it isn't, the practice notebooks already cover that.


### 1 — `SparkSession` has two versions

- `pyspark.sql.session.SparkSession` — **classic**, drives a local JVM.
- `pyspark.sql.connect.session.SparkSession` — **Connect**, talks to a remote Spark server over gRPC.

Same API surface, different classes. Helpers that need to touch the JVM directly (e.g. `delta.configure_spark_with_delta_pip(builder)`) accept only the classic builder and reject the Connect one with a type-mismatch error.


### 2 — Predicate pushdown: what does and doesn't push

- **In-memory sources can never push filters.** `Range` (`spark.range`), `LocalTableScan` / `ExistingRDD` (`createDataFrame`) have no scan layer to push into — the data is already in memory. The `Filter` just sits above the source.
- **Even when a filter pushes, the `Filter` node stays in the plan.** Source-level pushdown is best-effort (e.g. Parquet uses row-group min/max stats — it can skip whole groups but rows inside surviving groups still need exact re-checking). Catalyst keeps the `Filter` above the scan for correctness.

```text
*(1) Project [customer_id#0, full_name#1, credit_score#3L]
  +- *(1) Filter (isnotnull(credit_score#3L) AND (credit_score#3L >= 700))   ← exact re-check
     +- *(1) ColumnarToRow
        +- FileScan parquet [...]
             PushedFilters: [IsNotNull(credit_score), GreaterThanOrEqual(credit_score,700)]
             ReadSchema: struct<customer_id:string,full_name:string,credit_score:bigint>
```


### 3 — `foreach(print)` vs `show()`

Both are actions, but they print from different places and behave very differently:

| | `df.show()` | `df.foreach(print)` |
|---|---|---|
| Where the print happens | Driver | Executors |
| Where you see the output | Your console | Executor stdout / cluster logs — usually invisible in a notebook or `spark-submit --master yarn` |
| Output format | Pretty ASCII table, header + 20 rows by default | One `Row(...)` per line, unordered, no header |
| Order | First 20 rows in source order (top-down) | Whatever order partitions emit — interleaved across executors |
| Data movement | First N rows pulled to driver | None — runs on each row in place |
| Safe at scale | Yes (bounded) | Yes (no driver memory pressure), but log volume can be huge |

In `local[*]` mode the executor *is* the driver, so `foreach(print)` happens to land in the same console — which is why it looked equivalent to `show()` in the practice script. On a real cluster the prints disappear into executor logs and you'd think nothing happened.

Rule of thumb: use `show()` for inspecting data; reserve `foreach` for genuine per-row side effects (writing to a sink, pushing to a queue) and even then prefer `foreachPartition` to amortize connection setup.


### 4 — Temp views vs. tables

**Temp views — four flavours along two axes:**

|  | Errors if it exists | Replaces silently |
|---|---|---|
| **Session-scoped** | `createTempView(name)` | `createOrReplaceTempView(name)` |
| **Cross-session (application-scoped)** | `createGlobalTempView(name)` | `createOrReplaceGlobalTempView(name)` |

- Session views are visible only inside the current `SparkSession`. Global views live in a special `global_temp` database visible to every `SparkSession` in the same Spark application — both die when the application stops.
- Reference syntax: session view → `SELECT * FROM v`. Global view → `SELECT * FROM global_temp.v` (the prefix is mandatory).
- All four are **metadata only** — they register a name pointing at the DataFrame's logical plan. No data is written. Every query re-executes the upstream transformations (unless the underlying DataFrame is cached).
- Non-`OrReplace` variants throw `AnalysisException` when a name collides — useful as a guardrail when re-registering would silently be a bug.

**Tables — persistent in a catalog:**

| Kind | Created with | Data ownership | Drop behaviour | Lifetime |
|---|---|---|---|---|
| **Managed table** | `df.write.saveAsTable("t")` or `CREATE TABLE t … USING <fmt>` (no `LOCATION`) | Spark / the metastore owns the files | `DROP TABLE` deletes both metadata **and** the underlying files | Survives application restarts |
| **External table** | `df.write.option("path", "/loc").saveAsTable("t")` or `CREATE TABLE t … LOCATION '/loc'` | You own the files | `DROP TABLE` deletes only the metadata; files stay | Survives application restarts |

- Tables live in a catalog (Hive metastore, AWS Glue, Unity Catalog, …). Anything that can connect to the catalog can see them — across sessions, jobs, even applications.
- Managed table files are written under `spark.sql.warehouse.dir` by default. External tables go wherever you point them.
- A table can be backed by any format: Parquet, ORC, JSON, Delta, etc. The `USING delta` clause is what makes it a Delta table (and unlocks ACID, time travel, etc.).

**Rule of thumb:** use `createOrReplaceTempView` for scratch in notebooks; reach for a real table when results need to outlive the session, be queried from elsewhere, or get scheduled refreshes.


### 5 — `spark.stop()` vs `sc.stop()` — one is enough, both is harmless

`SparkSession` wraps a single underlying `SparkContext`. `spark.stop()` calls `sc.stop()` internally, so one call shuts both down.

- `spark.stop()` alone → enough. Conventional choice (session is the higher-level handle).
- `sc.stop()` alone → also enough. The session has no life without its context.
- Both, in either order → also fine, the second call is a silent no-op.

**Why no error on the second call?** `stop()` is **idempotent**. The JVM `SparkContext.stop()` does `stopped.compareAndSet(false, true)` first — if it's already `true`, the rest of the method is skipped. Designed that way because `stop()` is the kind of thing that ends up in `finally:` blocks, `atexit` handlers, signal handlers, JVM shutdown hooks, and IDE stop buttons — having shutdown itself raise would mask the real exit reason.

**The real guard is on use, not on close.** Calling `stop()` twice is fine; calling an *operation* on a stopped context is not:

```python
sc.stop()
sc.parallelize([1, 2, 3])   # IllegalStateException: SparkContext has been shut down
```


### 6 — RDD creation/transform options, grouped by purpose

Per-method option names are inconsistent (`numSlices` vs `minPartitions` vs `numPartitions`). Forgettable individually, easy once grouped by **what the option does**.

**Bucket 1 — Partition count:**

| Option | Appears on | Means |
|---|---|---|
| `numSlices` | `parallelize`, `range` | **Exact** partition count for driver-side collections |
| `minPartitions` | `textFile`, `wholeTextFiles`, `hadoopFile`, … | **Lower bound** for file reads; Spark may give more if files are large |

- Mnemonic: **driver collection → exact (`numSlices`); file read → at-least (`minPartitions`)**.
- **Default when omitted:** both fall back to `sc.defaultParallelism` (driven by `spark.default.parallelism` config — usually total cluster cores). So `sc.parallelize([1,2,3])` on a 12-core box creates 12 partitions for 3 elements — most empty. That's why "I see so many empty partitions" happens.

**Bucket 2 — The `path` parameter is more flexible than it looks**

Applies to `textFile`, `wholeTextFiles`, `binaryFiles`, all Hadoop readers. A single string accepts:

- A file: `"data/file.txt"`
- A directory (reads every file in it): `"data/"`
- A glob: `"data/*.csv"`, `"data/2024-{01,02}-*.json"`
- Comma-separated paths: `"data/a.txt,data/b.txt,other/c.txt"`
- Any supported URI scheme: `file://`, `hdfs://`, `s3a://`, `gs://`, `abfss://`, …

Skip the `glob.glob()` + Python `for` loop — the path parameter already does this.

**Bucket 3 — Decoding hazard on text readers**

| Option | Appears on | Means |
|---|---|---|
| `use_unicode` (default `True`) | `textFile`, `wholeTextFiles` | Decode bytes as UTF-8. Non-UTF-8 input (Latin-1, Shift-JIS) throws `UnicodeDecodeError` on the executor — surfaces as a cryptic task failure, not a parse error. Set `use_unicode=False` to get raw `bytes` and decode yourself. |

**Bucket 4 — Optimizer hints (assertions — wrong values corrupt silently):**

| Option | Appears on | Means |
|---|---|---|
| `preservesPartitioning` | `map`, `mapPartitions`, `flatMap`, `mapValues` | "My function does not change the keys, so a downstream `reduceByKey` / `join` can skip a shuffle." Only meaningful on a **pair RDD** (already partitioned by key). Lying here corrupts joins/aggregations with no error. |

- Mnemonic: **only set `preservesPartitioning=True` if your function literally cannot change the key** (e.g. `mapValues`, projection of unrelated fields).

**Retention rule:** when a new option shows up, ask *which bucket?* before recording it. New parameter on `sc.binaryFiles`? `minPartitions` again — bucket 1. New parameter on a `mapPartitions`-family method? Probably `preservesPartitioning` — bucket 4. Most "new" options are members of buckets you already know.

**Deliberately skipped** (real but rarely worth memorising): `recordLength` on `binaryRecords`, `keyClass` / `valueClass` / converters on `sequenceFile` / `hadoopFile`, `batchSize` on Hadoop readers. Look them up if you actually touch those APIs.


    ---

_New entries go below. Template:_

```
### N — title

One- or two-sentence observation. Code only if it adds something the practice notebooks don't already show.
```
